# Topic 4 — Memorystore (Redis): Cached AI Insights

Requires: `04a_provision_redis_and_bastion.bat` already run, AND the SSH tunnel from a second Command Prompt window still open — same pattern as Module 9's Redis topic.

In [ ]:
import redis
import time
import hashlib
from setup import genai_client, MODEL_FLASH

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

### The cache-or-call function

In [ ]:
def get_student_insight(student_id: str, question: str) -> str:
    cache_key = f"insight:{student_id}:{hashlib.md5(question.encode()).hexdigest()}"

    cached = r.get(cache_key)
    if cached:
        return f"[CACHE HIT] {cached}"

    response = genai_client.models.generate_content(model=MODEL_FLASH, contents=question)
    r.setex(cache_key, 3600, response.text)  # cache for 1 hour
    return f"[CACHE MISS] {response.text}"

### Run the same question twice — watch the timing

In [ ]:
question = "In 2 sentences, what does a typical grade trend look like for a B+ student improving steadily?"

start = time.time()
print(get_student_insight("alex", question))
print(f"\nTime: {time.time() - start:.2f}s\n")

In [ ]:
start = time.time()
print(get_student_insight("alex", question))
print(f"\nTime: {time.time() - start:.2f}s")